# tool_choice：控制 LLM 是否调用工具以及调用哪个工具
> `tool_choice` 是 `bind_tools` 的关键参数，决定 LLM 在工具调用时的行为：自动决定 / 强制调用 / 强制调用指定工具。

| 取值 | 含义 |
| --- | --- |
| `"auto"`（默认） | LLM 自己决定是否调用、调用哪个工具 |
| `"none"` | 不调用任何工具，强制 LLM 只生成普通文本回复 |
| `"required"` | 强制至少调用一个工具（具体调用哪个由 LLM 决定）|
| `"any"` | OpenAI 别名，语义同 `required` |
| `{"type": "function", "function": {"name": "工具名"}}` | 强制调用指定的工具 |
| `"工具名"`（LangChain 简写） | LangChain 也支持直接传工具名字符串 |

**注意**：`required` / `any` / 指定工具 这几种强制调用模式，OpenAI 旧模型（如 `gpt-3.5-turbo`）可能不支持，建议使用 `gpt-4o`、`gpt-4o-mini` 等较新模型。

## 0. 准备工作：定义工具和 LLM

In [ ]:
import os
import dotenv
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from rich import print as rprint

dotenv.load_dotenv()

# 定义两个工具：计算器 和 天气查询
@tool
def calculate(expression: str) -> str:
    """计算数学表达式，输入为数学表达式字符串，如 '2 + 3 * 4'。"""
    try:
        return f"计算结果：{expression} = {eval(expression)}"
    except Exception as e:
        return f"计算错误：{e}"

@tool
def get_weather(city: str) -> str:
    """根据城市名称查询当前天气信息。"""
    mock_weather = {
        "北京": "晴天，25°C",
        "上海": "多云，22°C",
        "广州": "小雨，28°C",
    }
    return mock_weather.get(city, f"未找到{city}的天气信息")

# 初始化 LLM
llm = ChatOpenAI(
    model=os.getenv("MODEL_NAME", "gpt-4o-mini"),
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL"),
    temperature=0,
)

tools = [calculate, get_weather]
rprint("[bold green]工具和 LLM 准备完毕[/bold green]", *[t.name for t in tools], sep="\n  - ")

## 1. `tool_choice="auto"`：LLM 自动决定（默认行为）

LLM 根据问题内容判断是否需要调用工具，调用哪个。

In [ ]:
# "auto" 是默认值，可写可不写
llm_auto = llm.bind_tools(tools, tool_choice="auto")

# 需要计算 → 调用 calculate
r1 = llm_auto.invoke("帮我算一下 (3 + 5) * 2")
rprint("[bold]问题1：帮我算一下 (3 + 5) * 2[/bold]")
rprint("  内容：", r1.content)
rprint("  工具调用：", r1.tool_calls)

# 问天气 → 调用 get_weather
r2 = llm_auto.invoke("北京天气怎么样？")
rprint("\n[bold]问题2：北京天气怎么样？[/bold]")
rprint("  内容：", r2.content)
rprint("  工具调用：", r2.tool_calls)

# 闲聊 → 不调用任何工具，直接回复文本
r3 = llm_auto.invoke("你好，介绍一下你自己")
rprint("\n[bold]问题3：你好，介绍一下你自己[/bold]")
rprint("  内容：", r3.content)
rprint("  工具调用：", r3.tool_calls)

## 2. `tool_choice="none"`：禁止调用任何工具

即使问题明显需要工具，LLM 也只能用文本回复，不会返回 `tool_calls`。

In [ ]:
llm_none = llm.bind_tools(tools, tool_choice="none")

# 同样的计算问题，但 LLM 不能调用工具，只能硬算或拒绝
r = llm_none.invoke("帮我算一下 123456 * 789")
rprint("[bold]问题：帮我算一下 123456 * 789[/bold]")
rprint("  内容：", r.content)
rprint("  工具调用：", r.tool_calls)
rprint("\n[i]可以看到 tool_calls 为空，LLM 只能用文本回复，无法调用计算器。[/i]")

## 3. `tool_choice="required"`：强制至少调用一个工具

LLM 必须调用工具，但具体调用哪个由 LLM 自己根据问题决定。

In [ ]:
# "any" 是 "required" 的 OpenAI 别名，二者效果相同
llm_required = llm.bind_tools(tools, tool_choice="required")

# 即使是闲聊，也必须调用工具（强制行为，演示用）
r1 = llm_required.invoke("你好，介绍一下你自己")
rprint("[bold]问题1：你好，介绍一下你自己（闲聊也强制调用工具）[/bold]")
rprint("  内容：", r1.content)
rprint("  工具调用：", r1.tool_calls)

# 正常的工具问题：LLM 选择合适的工具
r2 = llm_required.invoke("上海今天天气如何？")
rprint("\n[bold]问题2：上海今天天气如何？[/bold]")
rprint("  内容：", r2.content)
rprint("  工具调用：", r2.tool_calls)

## 4. `tool_choice={...}`：强制调用指定工具

无论问题是什么，LLM 都必须调用指定的工具。这是最精确的控制方式。

有两种写法：
- **OpenAI 标准格式**：`{"type": "function", "function": {"name": "工具名"}}`
- **LangChain 简写**：直接传工具名字符串 `"工具名"`

In [ ]:
# 写法一：OpenAI 标准格式，强制调用 calculate
llm_force_calc = llm.bind_tools(
    tools,
    tool_choice={"type": "function", "function": {"name": "calculate"}},
)

# 即使问天气，也强制用计算器（LLM 会硬塞一个表达式进去）
r1 = llm_force_calc.invoke("北京天气怎么样？")
rprint("[bold]写法一：OpenAI 标准格式，强制调用 calculate[/bold]")
rprint("  问题：北京天气怎么样？")
rprint("  内容：", r1.content)
rprint("  工具调用：", r1.tool_calls)

# 写法二：LangChain 简写，强制调用 get_weather
llm_force_weather = llm.bind_tools(tools, tool_choice="get_weather")

# 即使问数学题，也强制调用天气工具
r2 = llm_force_weather.invoke("帮我算一下 1 + 1")
rprint("\n[bold]写法二：LangChain 简写，强制调用 get_weather[/bold]")
rprint("  问题：帮我算一下 1 + 1")
rprint("  内容：", r2.content)
rprint("  工具调用：", r2.tool_calls)

## 5. 实战：强制调用指定工具并自动执行

一个常见场景：用户问的可能是任意问题，但我们希望**必定经过某个工具**（比如必定查询数据库、必定搜索知识库），这时强制指定工具就很有用。

In [ ]:
from langchain_core.messages import AIMessage, ToolMessage

# 场景：无论用户问什么，都强制走 get_weather 工具
llm_force = llm.bind_tools(tools, tool_choice="get_weather")

tool_map = {"calculate": calculate, "get_weather": get_weather}

def force_weather_query(question: str) -> str:
    """强制走天气工具的完整流程：调用工具 → 拿结果 → LLM 总结。"""
    rprint(f"\n[bold cyan]用户问题：{question}[/bold cyan]")

    messages = [HumanMessage(content=question)]

    # 第一步：LLM 被强制调用 get_weather
    ai_resp: AIMessage = llm_force.invoke(messages)
    rprint("  LLM 工具调用：", ai_resp.tool_calls)
    messages.append(ai_resp)

    # 第二步：手动执行工具
    for tc in ai_resp.tool_calls:
        result = tool_map[tc["name"]].invoke(tc["args"])
        rprint(f"  工具 {tc['name']}({tc['args']}) → {result}")
        messages.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))

    # 第三步：把工具结果交回 LLM 做最终总结（此时用 auto，不再强制）
    final = llm.bind_tools(tools, tool_choice="auto").invoke(messages)
    rprint("  [bold green]最终回答：[/bold green]", final.content)
    return final.content

# 演示三个问题
force_weather_query("北京和上海天气如何？")
force_weather_query("今天适合出门吗？")
force_weather_query("帮我算一下 2+2")  # 即使是计算问题，也会被强制走天气工具

## 6. 总结对比

| 场景 | `tool_choice` 取值 | 行为 |
| --- | --- | --- |
| 让 LLM 自己判断 | `"auto"`（默认）| 该调用就调用，不该调用就文本回复 |
| 完全禁用工具 | `"none"` | 永不调用工具，纯文本对话 |
| 必须用工具，但让 LLM 选 | `"required"` / `"any"` | 强制调用至少一个工具 |
| 必须用某个指定工具 | `{"type":"function","function":{"name":"X"}}` 或 `"X"` | 强制调用工具 X |

**使用建议**：
- 生产环境大多数情况用默认的 `"auto"` 即可
- 需要确保某个动作一定发生（如必查知识库、必写日志）时，用强制指定工具
- 需要纯对话模式（如闲聊分支）时，用 `"none"` 关闭工具调用避免误触发
- `"required"` 适合多工具场景下要求 LLM 必须采取行动（如 Agent 的某些步骤）